# 가중 사건 확률 계산기

작은 선택 시스템의 결과를 확률로 다루려면 먼저 무엇을 한 결과로 셀지 정하고, 각 원시 결과가 같은 확률인지 확인해야 합니다. 이 실습에서는 순서를 지운 선택의 수를 계산하고, 가중된 원시 결과의 사건 확률과 겹치는 두 사건의 합집합 확률을 구현합니다.

위에서 아래 순서대로 구현한 뒤 각 fixture와 검사 셀을 실행하세요. 마지막에는 경우의 수 비율이 통하는 조건과 겹침을 한 번만 빼는 이유를 결과에 연결합니다.


In [1]:
# 공통 준비
import math
import numpy as np


## 1. 순서를 지운 선택 세기

### 상황

`total`개의 서로 다른 후보 중 `selected`개를 고를 때, 역할이나 순서를 기록하지 않으면 `{A, B}`와 `{B, A}`는 같은 선택입니다. 이 함수는 그런 선택의 수를 반환합니다.

### 구현할 계약

- `choose_count`는 `total`개 중 순서를 구분하지 않고 `selected`개를 고르는 수를 반환해야 합니다.
- 경우의 수 비율을 확률로 쓰려면 이 선택들이 아니라, 그 선택을 만든 원시 결과가 서로 같은 확률인지 별도로 확인해야 합니다.
- `total`이 음수이거나 `selected`가 `0`과 `total` 사이가 아니면 `ValueError`로 거부해야 합니다.

### 작은 예

후보가 `A`, `B`, `C`, `D` 네 개이고 두 명을 고르면 가능한 집합은 여섯 개입니다. `A`를 포함하는 집합은 `AB`, `AC`, `AD` 세 개입니다.

<details><summary>힌트 1</summary>

먼저 순서를 구분하는 선택 수를 세고, 같은 집합으로 합쳐진 순열 수로 나눌 수 있습니다.

</details>

<details><summary>힌트 2</summary>

`math.factorial`로 분자와 분모를 만들되, `selected`가 `0`일 때도 자연스럽게 동작하는지 확인하세요.

</details>


In [21]:
def choose_count(total: int, selected: int) -> int:
    """Return the number of unordered selections of selected items."""
    if total < 0 or not 0 <= selected <= total:
        raise ValueError("selected must be between 0 and total")

    # TODO: 순서를 지운 선택의 분모를 완성하세요.
    denominator = math.factorial(total - selected) * math.factorial(selected)
    return math.factorial(total) // denominator


In [22]:
print(choose_count(4, 2))


6


In [23]:
def check_e01() -> None:
    np.testing.assert_equal(choose_count(4, 2), 6)
    np.testing.assert_equal(choose_count(5, 0), 1)

    try:
        choose_count(4, 5)
    except ValueError:
        pass
    else:
        raise AssertionError("selected가 total보다 크면 ValueError여야 합니다")


check_e01()


### 결과 해석

아래 빈칸에 2~4문장으로 적으세요. `choose_count(4, 2) = 6`을 확률 분모로 쓸 수 있는 정확한 조건과, 어떤 가중 선택 규칙에서는 여섯 집합을 하나씩 세면 안 되는 이유를 설명하세요.

<!-- TODO: 경우의 수 비율이 확률이 되는 조건을 해석하세요. -->
6을 확률 분모로 써도 되는 조건: 여섯 unordered pair 자체가 같은 확률로 선택되거나, 원시 결과를 합쳤을 때 각 pair의 확률이 같아야 함
가중 규칙에서는 pair별 확률이 달라질 수 있으므로 유리한 pair 수 / 6 대신 해당 원시 결과들의 가중치를 합해야 함


## 2. 가중 원시 결과에서 사건 확률 구하기

### 상황

이제 원시 결과마다 다른 확률을 줄 수 있습니다. `weights`는 각 결과의 확률을 저장한 dict이고, `event`는 그 결과들 중 관심 있는 결과의 set입니다. 사건의 확률은 event에 속한 원시 결과 확률의 합입니다.

### 구현할 계약

- `event_probability`는 event에 포함된 각 원시 결과의 확률을 더해 사건 확률을 반환해야 합니다.
- `weights`의 값은 모두 `0` 이상이어야 하고 전체 합은 `1.0`이어야 하며, event에 weights에 없는 결과가 있으면 `ValueError`로 거부해야 합니다.

### 작은 예

`{"AA": 0.5, "AB": 0.25, "BA": 0.25}`에서 `{"AA", "AB"}` 사건의 확률은 `0.75`입니다. 원시 결과를 같은 개수로 세는 대신 각각의 가중치를 더한 것입니다.

<details><summary>힌트 1</summary>

빈 사건은 어떤 원시 결과도 포함하지 않으므로 합이 `0`이 됩니다.

</details>

<details><summary>힌트 2</summary>

먼저 event의 모든 원소가 weights의 key인지 확인한 뒤, 그 원소들에 대한 값을 합산하세요.

</details>


In [24]:
def event_probability(weights: dict[str, float], event: set[str]) -> float:
    """Return the probability mass assigned to one event."""
    if not weights or any(weight < 0 for weight in weights.values()):
        raise ValueError("weights must be nonempty and nonnegative")
    if not np.isclose(sum(weights.values()), 1.0):
        raise ValueError("weights must sum to 1.0")
    if not event.issubset(weights):
        raise ValueError("event contains an unknown outcome")

    # TODO: event에 속한 원시 결과의 가중치를 합산하세요.
    probability = 0

    for v in event:
        probability += weights[v]

    return float(probability)


In [25]:
sample_weights = {"AA": 0.5, "AB": 0.25, "BA": 0.25}
print(event_probability(sample_weights, {"AA", "AB"}))


0.75


In [26]:
def check_e02() -> None:
    weights = {"AA": 0.5, "AB": 0.25, "BA": 0.25}
    np.testing.assert_allclose(event_probability(weights, {"AA", "AB"}), 0.75)
    np.testing.assert_allclose(event_probability(weights, set()), 0.0)

    try:
        event_probability(weights, {"missing"})
    except ValueError:
        pass
    else:
        raise AssertionError("알 수 없는 원시 결과는 ValueError여야 합니다")


check_e02()


### 확인 결과 정리

선택 복습입니다. 원시 결과마다 확률이 다를 때 `event_probability`가 왜 경우의 수 비율 대신 가중치 합을 쓰는지 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 3. 겹치는 사건의 합집합 보정하기

### 상황

두 사건 `event_a`, `event_b`가 겹치면 교집합 결과의 확률이 각각의 사건 확률에 한 번씩 들어갑니다. 따라서 합집합 확률에서는 그 교집합을 한 번 빼야 합니다.

### 구현할 계약

- `union_probability`는 `event_a` 확률과 `event_b` 확률을 더한 뒤 두 사건의 교집합 확률을 한 번 빼서 반환해야 합니다.
- 두 사건 중 하나에 weights에 없는 결과가 있으면 `ValueError`로 거부해야 합니다.
- 결과 해석에는 교집합이 두 번 더해진 뒤 한 번 빼지는 이유와, 가중된 원시 결과에서는 단순 경우의 수 비율을 쓰지 않는 이유를 함께 써야 합니다.

### 작은 예

E02의 weights에서 `A = {"AA", "AB"}`와 `B = {"AB", "BA"}`는 `AB`에서 겹칩니다. 각각의 확률을 그냥 더하면 `AB`의 `0.25`가 두 번 들어갑니다.

<details><summary>힌트 1</summary>

교집합은 Python set의 `&` 연산으로 만들 수 있습니다.

</details>

<details><summary>힌트 2</summary>

이미 만든 `event_probability`를 세 번 호출해 각 항을 계산하면 됩니다.

</details>


In [28]:
def union_probability(weights: dict[str, float], event_a: set[str], event_b: set[str]) -> float:
    """Return the probability of event_a or event_b, counting overlap once."""
    probability_a = event_probability(weights, event_a)
    probability_b = event_probability(weights, event_b)
    overlap_probability = event_probability(weights, event_a & event_b)

    # TODO: 두 사건의 교집합 확률을 한 번만 남기세요.
    probability = probability_a + probability_b - overlap_probability
    return float(probability)


In [29]:
sample_weights = {"AA": 0.5, "AB": 0.25, "BA": 0.25}
print(union_probability(sample_weights, {"AA", "AB"}, {"AB", "BA"}))


1.0


In [30]:
def check_e03() -> None:
    weights = {"AA": 0.5, "AB": 0.25, "BA": 0.25}
    np.testing.assert_allclose(union_probability(weights, {"AA", "AB"}, {"AB", "BA"}), 1.0)
    np.testing.assert_allclose(union_probability(weights, {"AA"}, {"BA"}), 0.75)

    try:
        union_probability(weights, {"AA"}, {"missing"})
    except ValueError:
        pass
    else:
        raise AssertionError("알 수 없는 원시 결과는 ValueError여야 합니다")


check_e03()


### 결과 해석

아래 빈칸에 2~4문장으로 적으세요. `AB`가 왜 두 사건 확률을 더할 때 두 번 들어가는지, 왜 한 번만 빼면 되는지, 그리고 이 가중 예에서 집합 수의 비율만으로 확률을 계산할 수 없는 이유를 설명하세요.

<!-- TODO: 겹침 보정과 가중 확률의 관계를 해석하세요. -->
A사건을 구할 때, B사건을 구할 때 각각 AB가 들어가므로 2번 들어간다.
따라서 중복되는 1번만 빼면 된다.
각 사건에 대한 가중이 다르므로 비율만으로 확률을 계산할 수 없다.
